# T03. Tokens become a tree

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/t03-tokens-become-a-tree/t03.ipynb)

T02 left you with a flat list of [tokens](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#token): a name, an equals sign, a number, a star, another number. That list is in the right order, and it says nothing about what goes with what.

This lesson is about the part that fixes that. The parser reads the tokens and builds an [abstract syntax tree](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#abstract-syntax-tree), and the tree is the first thing in the pipeline that knows `6 * 7` is one thing rather than three.

![the eight stages of running Python, with the syntax tree highlighted](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t03-tokens-become-a-tree/diagrams/where-we-are.svg)

The interesting question about the tree is not how it gets built but what it keeps. Your brackets are gone by the end of this stage, and so are your spacing and your comments, and everything they meant is still there. By the end you will have watched three different files turn into exactly the same tree, and then checked that claim against every module in your own standard library.

No C required and no build of your own, since everything here runs on a normal Python.

## About the source references

Now and then this lesson points at CPython's own source, like this: `Parser/pegen.c:938-941@v3.15.0rc1#_PyPegen_run_parser`.

Read it as four parts: the file, the lines, the release those line numbers belong to, and the name of the function they are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. The function name on the end is what makes the check work. Line numbers move whenever somebody adds code above them, and a moved line number points at something that looks plausible and is not.

You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Node types get added between releases and the tree for a given piece of code can change shape, so every lesson here starts by saying which build produced the output you are about to read.

In [ ]:
import pyxray

pyxray.show()

## One line, one tree

`ast.parse` runs the real parser, the same one that runs when you import a module. It is a [PEG parser](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#peg-parser), generated from a [grammar](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#grammar) file into `Parser/parser.c` and driven by [Parser/pegen.c:938-941@v3.15.0rc1#_PyPegen_run_parser](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/pegen.c#L938-L941), and the Python side of it is [Lib/ast.py:26-30@v3.15.0rc1#parse](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/ast.py#L26-L30), which is a thin wrapper around `compile` with a flag set.

Start with the line from T01, where the tree is the first thing in the pipeline that knows `6 * 7` is one thing rather than three, with both numbers hanging underneath a single BinOp.

In [ ]:
from pyxray import trees

SOURCE = "answer = 6 * 7\n"

print(trees.outline(SOURCE))

Read it from the inside out. Two `Constant` nodes hold 6 and 7. A `BinOp` holds those two with a `Mult` between them. An `Assign` puts the result into a `Name` whose `ctx` is `Store`, meaning this name is being written to rather than read from.

The indented view above is this project's, and it is there because the shape is the point. CPython's own view is [Lib/ast.py:117-121@v3.15.0rc1#dump](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/ast.py#L117-L121), which prints everything and is what you want when you need to be exact rather than quick.

In [ ]:
import ast

print(ast.dump(ast.parse(SOURCE), indent=4))

## Where the node types are written down

Python's syntax tree is not defined inside a compiler somewhere. It is defined in one readable file, [Parser/Python.asdl:62@v3.15.0rc1#BinOp](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/Python.asdl#L62), in a small language called [ASDL](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#asdl) that exists to describe tree shapes.

The line for `BinOp` in that file says it has three fields: an expression on the left, an operator, and an expression on the right. The neat part is that the same line ends up as the docstring of the class.

![Python.asdl is read by asdl_c.py, which generates the classes and puts the declaration in the docstring](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t03-tokens-become-a-tree/diagrams/where-the-node-classes-come-from.svg)

[Parser/asdl_c.py:95-109@v3.15.0rc1#asdl_of](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/asdl_c.py#L95-L109) turns each declaration back into text at build time, and [Parser/asdl_c.py:1617-1619@v3.15.0rc1#make_type](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/asdl_c.py#L1617-L1619) hands it to `type()` as the docstring of the generated class. So the ASDL declaration of a node type is carried around as the docstring of the class, which means printing it gives you the definition itself rather than somebody's description of it.

In [ ]:
print(trees.asdl(ast.BinOp))
print(trees.fields(ast.BinOp))

`expr left` means the left side is any expression at all, not just a number. That single word is why `1 + 2 * 3` can nest: the left of a `BinOp` is allowed to be another `BinOp`, and so is the right.

the operators are a closed list of cases rather than nodes with fields, so Mult has no fields at all, and asking for the declaration gives you every operator Python has.

In [ ]:
print(trees.asdl(ast.operator))
print()
print("Mult has fields:", trees.fields(ast.Mult))

`Mult` has no fields, because it is a case rather than a container. It is not a node holding a `*` somewhere inside it, it is the name of which operator this is, and the full list is fixed at [Parser/Python.asdl:104-105@v3.15.0rc1#operator](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/Python.asdl#L104-L105). An operator that is not on that list cannot reach the compiler, because there is nothing for the parser to build.

The C function that actually assembles one of these is [Python/Python-ast.c:7767-7770@v3.15.0rc1#_PyAST_BinOp](https://github.com/python/cpython/blob/v3.15.0rc1/Python/Python-ast.c#L7767-L7770), and it is generated from the same ASDL file.

## Three files, one tree

The claim this lesson is really about is that the tree keeps what your code means and throws away how you wrote it.

![three differently written files all producing one identical tree](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t03-tokens-become-a-tree/diagrams/three-sources-one-tree.svg)

Three files, with brackets in one, no spaces in another and a comment on the third. brackets, spacing and a trailing comment make no difference to the tree: three differently written files give one identical tree, field for field.

In [ ]:
WRITTEN = ["answer = (6 * 7)", "answer=6*7", "answer = 6 * 7  # note"]

for other in WRITTEN[1:]:
    print(f"{other!r:<28} same tree as {WRITTEN[0]!r}? {trees.same_tree(WRITTEN[0], other)}")

All three give the same tree, and not just a similar or equivalent one: identical, field for field.

It is worth seeing that the brackets really were there a moment ago. the tokenizer hands the parser both brackets as ordinary tokens, and the parser is where they stop existing.

In [ ]:
from pyxray import compiler

for item in compiler.tokens("answer = (6 * 7)\n"):
    if item.string.strip():
        print(f"{item.string!r}", end="  ")
print()
print()
print(trees.outline("answer = (6 * 7)\n"))

## What the brackets did is still there

The obvious objection is that brackets change what code means, so they cannot just vanish. They do not vanish, they are turned into shape.

![the trees for 1 + 2 * 3 and (1 + 2) * 3, side by side](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t03-tokens-become-a-tree/diagrams/precedence-is-the-shape.svg)

the same five tokens in the same order give two different trees, because what the brackets did is kept as shape. On the left the multiply is inside the add, because `2 * 3` has to happen first. On the right it is the other way round.

In [ ]:
print(trees.outline("x = 1 + 2 * 3"))
print()
print(trees.outline("x = (1 + 2) * 3"))
print()
print("same tree?", trees.same_tree("x = 1 + 2 * 3", "x = (1 + 2) * 3"))

The precedence is not stored anywhere as a number. It is in the grammar, and it comes out as nesting. [Grammar/python.gram:841-844@v3.15.0rc1#sum](https://github.com/python/cpython/blob/v3.15.0rc1/Grammar/python.gram#L841-L844) says a `sum` is a `sum` plus a `term`, and [Grammar/python.gram:846-852@v3.15.0rc1#term](https://github.com/python/cpython/blob/v3.15.0rc1/Grammar/python.gram#L846-L852) says a `term` is a `term` times a `factor`. Because a sum is built out of terms and not the other way round, multiplication ends up further down the tree, which is exactly what "binds tighter" means.

Those rules are also where the node gets built. Look at the right hand side of the grammar line for `*` and you will see `_PyAST_BinOp(a, Mult, b, EXTRA)`, which is the C constructor from earlier being called by the parser.

## Every node remembers where it came from

The tree forgets your formatting, and it does keep track of which characters each node came from. That is how a traceback can point at a column, and how tools like linters can tell you where the problem is.

[Lib/ast.py:385-390@v3.15.0rc1#get_source_segment](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/ast.py#L385-L390) goes the other way, from a node back to the text it covers.

Two things in the list below. every node that was written somewhere carries the line and column range it covers, and a node with no fields has no position either, so there is nothing in the tree that says where the `*` was.

In [ ]:
for span in trees.spans(SOURCE):
    print(span)

`Mult` is missing from that list, and that is not a bug. Nodes with no fields have no position either, because `Mult` is not something written at a place in the file. It is which case this `BinOp` is. If you go looking for the position of an operator you will not find one.

## Turning a tree back into text

[Lib/ast.py:653-659@v3.15.0rc1#unparse](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/ast.py#L653-L659) takes a tree and gives you source code back. It is not your source code, it is source code that means the same thing.

![a table of five things you might write and what unparse gives back](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t03-tokens-become-a-tree/diagrams/what-unparse-rewrites.svg)

Every row of that table is a different file and the same tree, so `unparse` has nothing to go on when it decides how to write it out. It picks one way and uses it every time, which is why unparse gives back source that means the same thing rather than the source you wrote: 0x2a comes back as 42 and the underscores in 1_000_000 are gone. adjacent string literals are glued together by the grammar, so 'a' 'b' is already one string before anything runs.

In [ ]:
for source in ["x = 0x2a", "x = 1_000_000", "x = 'a' 'b'", "x = (1 + 2)", "x = 1  # note"]:
    print(f"{source:<22} {trees.roundtrip(source)}")

`0x2a` comes back as `42` because the tree holds the number and not the base you wrote it in. `1_000_000` loses its underscores for the same reason. `'a' 'b'` was already joined into one string by the parser, since adjacent string literals are glued together as part of the grammar rather than at runtime.

Every one of them says "same tree", which is the property worth remembering and worth checking rather than believing.

## The round trip, on real code

The property in one sentence: parse a file, unparse it, parse the result, and you get an identical tree, for every module in your own standard library.

Four examples do not prove a property, so run it over every module that shipped with the interpreter you are using right now.

In [ ]:
report = trees.survey(trees.stdlib())

print("standard library at", trees.stdlib())
print(report)

> **Version note.** These numbers describe the Python you are running rather than the language, so they will not match the text exactly. A framework install, a source build and a Colab image all count different files.

Every module gives the same tree both times. That is a real property test over a few hundred thousand lines of code that nobody wrote for this lesson, and it took about a second.

It is worth being clear about what it does not prove. It says nothing about whether the unparsed text is nice to read, and nothing about the files that were skipped, which are mostly test fixtures that are deliberately not valid on this version. What it does show is that the tree really is the whole meaning of the file, because you can throw the file away and rebuild it.

## Try it yourself

Change `MINE` below and run the cell. Things worth trying, roughly in order of how surprising the answer is.

Try `x = 1 if a else 2` and look at where the condition ends up in the tree. It is not first, even though you typed it in the middle.

Try `def f(a, b=1, *args, **kw): pass` and look at the `arguments` node. Every kind of parameter Python has is a separate field, which is why the tree is a better thing to write a tool against than the text.

Try `x = -5` and count the nodes. The minus sign is a `UnaryOp` wrapped around a `Constant 5`, so there is no negative number literal in Python at all.

Try `f'{a + b}'` and watch the code inside the braces come out as an ordinary tree of its own, which is the tokenizer behaviour from T02 followed through to this stage.

Try something with a syntax error, like `x = (1 +`, and see what the parser says.

In [ ]:
MINE = "x = 1 if a else 2"

print(trees.outline(MINE))
print()
print(trees.roundtrip(MINE))

## What just happened

A flat list of tokens became a tree that knows what goes with what. The tree dropped your brackets, your spacing and your comments, and kept every bit of what they meant by putting it into the shape. Each node type is declared in one file, in a small language for describing tree shapes, and carries that declaration around as its own docstring. Every node that was written somewhere remembers where.

Then you checked the whole thing by turning trees back into text and parsing them again, on real code, and got the same tree every time.

## Where this goes next

The tree is the last stage that is only about syntax. Everything after it is about meaning.

T04 is the next box along, and it is the first pass that asks a question the tree cannot answer on its own: when this code says `answer`, which `answer` is that? The tree has a `Name` node and nothing else. The [symbol table](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#symbol-table) decides whether that name is a local, a global, or something borrowed from an enclosing function, and the compiler cannot pick an instruction until it knows.

That pass is also where one of Python's most confusing error messages comes from. A function that assigns to a name anywhere treats it as local everywhere, including on the line before the assignment, which is why you can get an `UnboundLocalError` from a name that clearly has a value.